# 4. Exports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/04_exports.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/04_exports.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

You export the network of Monaco to six formats. Each cell writes the files and does a check on them.

| Format | Call | Check |
|---|---|---|
| SUMO | `net.to_sumo` (`duckosm.sim`) | SUMO's `netconvert` reads the plain XML and writes `.net.xml` |
| GMNS | `net.to_gmns` (`duckosm.planning`) | `duckosm.planning.check_gmns` |
| MATSim | `net.to_matsim` (`duckosm.sim`) | the XML has one link for each edge |
| OpenDRIVE | `net.to_opendrive` (`duckosm.hdmap`) | the XML has one road for each edge |
| GeoPackage | `net.to_gis` (`duckosm.gis`) | DuckDB reads each layer |
| QuackOSM table, PMTiles | `duckosm.gis.quack` | DuckDB reads the table; the PMTiles header |

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
try:                       # duckosm is there: no install
    import duckosm         # the SDK
except ImportError:        # else install it with the extras of this notebook
    %pip install -q "duckosm[widget,viz,sim]"
    import duckosm         # the SDK
print("duckosm", duckosm.__version__)    # the version

duckosm 0.8.0


In [2]:
net = duckosm.network(osm_id=2220322, modes=["driving"], name="monaco")    # get and build the driving network of Monaco
net.summary()                                         # the number of edges, to compare with each export

cut: 34461 nodes, 5685 ways, 339 relations

Output: data/monaco.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5


## SUMO

`to_sumo` writes the plain XML files and runs SUMO's `netconvert` on them. The extra `sim` installs `netconvert`.

In [3]:
sumo = net.to_sumo("out/sumo")    # the plain XML files, and the .net.xml of netconvert
sumo                              # the paths and the numbers

SUMO out/sumo: done in 0.6 s

{'nod': PosixPath('out/sumo/network.nod.xml'),
 'edg': PosixPath('out/sumo/network.edg.xml'),
 'n_nodes': 1768,
 'n_edges': 2840,
 'n_connections': 5082,
 'con': PosixPath('out/sumo/network.con.xml'),
 'netccfg': PosixPath('out/sumo/network.netccfg'),
 'net': PosixPath('out/sumo/network.net.xml')}

## GMNS

In [4]:
net.to_gmns("out/gmns.duckdb", to_csv="out/gmns")                 # the GMNS tables in a DuckDB file and as CSV
checks = duckosm.planning.check_gmns("out/gmns.duckdb")             # (schema, check, bad rows, rows) for each check
len(checks), [c for c in checks if c[2] > 0]                        # the number of checks, and the checks with bad rows

GMNS out/gmns.duckdb: done in 6.6 s

(17, [])

## MATSim

In [5]:
import gzip
import xml.etree.ElementTree as ET                                   # to read the XML back

matsim = net.to_matsim("out/matsim_network.xml.gz")                  # the MATSim network
matsim, len(ET.parse(gzip.open("out/matsim_network.xml.gz")).findall(".//link"))    # its numbers, and the links in the file

MATSim out/matsim_network.xml.gz: done in 0.1 s

({'nodes': 1768, 'links': 2840, 'crs': 'EPSG:32632'}, 2840)

## OpenDRIVE (experimental)

In [6]:
odr = net.to_opendrive("out/monaco.xodr")                    # the OpenDRIVE file
odr, len(ET.parse("out/monaco.xodr").findall("road"))       # its numbers, and the roads in the file

OpenDRIVE out/monaco.xodr: done in 0.1 s

({'roads': 2840, 'crs': 'EPSG:32632'}, 2840)

## GeoPackage

In [7]:
import duckdb                                                # to read the layers back

gis = net.to_gis("out/monaco.gpkg")                          # the GeoPackage
con = duckdb.connect()                                       # DuckDB with the extension spatial
con.load_extension("spatial")
{layer: con.execute(f"SELECT count(*) FROM st_read('out/monaco.gpkg', layer='{layer}')").fetchone()[0]
 for layer in gis["layers"]}                                 # the rows of each layer, as DuckDB reads them

GIS out/monaco.gpkg: done in 0.2 s

{'edges_driving': 2840, 'nodes_driving': 1768, 'boundary': 1}

## QuackOSM table and PMTiles

`quack` reads the OSM file, not the network: it gives all OSM features with their tags, as QuackOSM does. The command `duckosm quack --pmtiles` makes a base map of vector tiles in one file.

In [8]:
area = duckosm.get(osm_id=2220322, name="monaco", out="data")                                          # the OSM file (from the cache)
quack = duckosm.gis.quack(area, out="out", name="monaco", formats=["geoparquet"])    # the QuackOSM table
con.sql(f"SELECT split_part(feature_id, '/', 1) AS type, count(*) AS features FROM '{quack[0]}' GROUP BY 1 ORDER BY 1").df()    # the features by OSM type

cut: 34461 nodes, 5685 ways, 339 relations

out/monaco_quack.parquet

,type,features
0,node,3994
1,relation,59
2,way,5606


In [9]:
import subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "duckosm", "quack", str(area.pbf), "--pmtiles", "-o", "out", "--name", "monaco"],
               check=True)                                             # the command duckosm quack --pmtiles
Path("out/monaco.pmtiles").read_bytes()[:7]                             # a PMTiles file starts with "PMTiles"

b'PMTiles'